# Chapter 11. Segmental-measure audit

**Learning objective.** Reconstruct VOT, closure, vowel duration, formants, and fricative measures from stable events while retaining absent, ambiguous, and invalid requests.

**Estimated time:** 75–90 minutes. **Exercises:** 11.1 and 11.2. The 1,080 tokens, event times, and acoustic estimates are explicitly **synthetic teaching fixtures** and do not describe speech.

## Input and output contract

Inputs are the event dictionary, token annotation manifest, acoustic estimates, and requested-measure manifest. Outputs are one raw row per request, a landmark audit, contextual coverage, HTML diagnostics, analysis decisions, and provenance under `companion/outputs/ch11/01_segmental_measure_audit/`. No failure is removed silently.

In [ ]:
find_repository <- function(start = getwd()) {
  current <- normalizePath(start, mustWork = TRUE)
  repeat {
    if (file.exists(file.path(current, "companion", "data", "MANIFEST.tsv"))) return(current)
    parent <- dirname(current)
    if (identical(parent, current)) stop("Run from the repository root or a subdirectory.")
    current <- parent
  }
}
REPOSITORY_ROOT <- find_repository()
DATA_ROOT <- file.path(REPOSITORY_ROOT, "companion", "data", "ch11", "synthetic_segmental")
if (!dir.exists(DATA_ROOT)) stop("Chapter 11 fixture absent. Run companion/scripts/generate_ch11_fixture.py explicitly; no data are substituted silently.")
source(file.path(REPOSITORY_ROOT, "companion", "r", "ch11.R"))
cat("DATA STATUS: SYNTHETIC_TEACHING_FIXTURE — generated landmarks and measures, not speech observations.
")

In [ ]:
OUTPUT <- file.path(REPOSITORY_ROOT, "companion", "outputs", "ch11", "01_segmental_measure_audit")
result <- run_segmental_measure_audit(file.path(DATA_ROOT, "event_dictionary.yaml"), file.path(DATA_ROOT, "annotation_manifest.tsv"), file.path(DATA_ROOT, "acoustic_estimates.tsv"), file.path(DATA_ROOT, "segmental_measure_manifest.tsv"), OUTPUT)
stopifnot(nrow(result$raw) == 3240)
stopifnot(nrow(result$audit) > 0)
stopifnot(all(c("vot", "closure_duration", "vowel_onset_f0", "vowel_duration", "f1", "f2", "f3", "fricative_duration", "fricative_cog") %in% result$raw$measure_id))
print(result$coverage)
cat("Retained review or failure rows:", nrow(result$audit), "
")

## Learner gate

Inspect landmark order, absent periodicity, multiple releases, context coverage, and the relationship among signed VOT, closure, and vowel-onset F0. Decide **accept**, **restrict**, **amend and rerun**, or **reject** separately for each requested measure before fitting any outcome model.

In [ ]:
learner_gate <- list(data_status = "SYNTHETIC_TEACHING_FIXTURE", decision_status = "undecided", measure_decisions = list(), rationale = "Learner response required.", warning = "Generated cue profiles are not empirical laryngeal contrasts.")
jsonlite::write_json(learner_gate, file.path(OUTPUT, "decision_record.json"), pretty = TRUE, auto_unbox = TRUE)
stopifnot(all(file.exists(file.path(OUTPUT, c("segmental_raw_measures.tsv", "landmark_audit.tsv", "coverage_by_context.tsv", "segmental_diagnostics.html", "analysis_decisions.yaml", "run_provenance.json", "decision_record.json")))))
print(learner_gate)